# Flatten & Reshape in Transformers

Reshaping tensors — merging or splitting dimensions — shows up everywhere in transformer code. Two places you'll always see it: **multi-head attention** and **loss computation over logits**.

The core idea is always the same: reshape doesn't move or change any values, it only changes how the same underlying data is grouped into dimensions.

```python
v = torch.arange(500)     # shape (500,)
m = v.view(2, 250)        # same 500 numbers, just read as (2, 250)
```

There are two directions this goes:

- **Flatten (merge)**: collapse two or more dimensions into one.
- **Split (unflatten)**: break one dimension into two or more.

---

## Flatten: Logits + Loss

The model outputs logits of shape `(B, T, vocab_size)` — a batch of `B` sequences, each with `T` timesteps, each timestep holding a score for every word in the vocabulary.

`CrossEntropyLoss` expects a much simpler shape: `(N, C)`, where each row is one classification example and each column is one candidate class.

So every token, from every sequence in the batch, gets treated as an **independent classification example**: "given the context so far, which token comes next?" There are `B * T` such examples in total. Flattening merges the batch and time dimensions into one:

```python
logits = logits.view(-1, vocab_size)   # (B, T, vocab_size) -> (B*T, vocab_size)
labels = labels.view(-1)               # (B, T) -> (B*T,)

loss = F.cross_entropy(logits, labels)
```

Row `i` of the flattened logits and element `i` of the flattened labels always correspond to the same token, because both were flattened in the same order (batch-major, then time). Nothing about the sequence structure matters to `cross_entropy` — it just sees `B*T` independent rows.

---

## Split: Multi-Head Attention

Attention does the opposite operation. After the Q/K/V projections, you have tensors of shape `(B, T, C)`, where `C` is the model dimension (e.g. 768). To run attention with multiple heads in parallel, `C` needs to be **split** into `(n_heads, head_dim)`:

```python
# (B, T, C) -> (B, T, n_heads, head_dim)
q = q.view(B, T, n_heads, head_dim)

# bring heads next to batch: (B, n_heads, T, head_dim)
q = q.transpose(1, 2)
```

Each head now attends independently over its own slice of the channel dimension.

After computing attention, the heads need to be **merged back** into `(B, T, C)` before the output projection:

```python
# (B, n_heads, T, head_dim) -> (B, T, n_heads, head_dim)
out = out.transpose(1, 2)

# merge heads back: (B, T, C)
out = out.reshape(B, T, n_heads * head_dim)
```

---

## `view` vs `reshape` — the contiguity issue

`transpose` doesn't move data in memory — it just changes the strides (how dimensions map to memory offsets). This makes the tensor **non-contiguous**.

- `.view()` requires a contiguous tensor. Calling it right after `.transpose()` will throw an error.
- `.reshape()` handles this automatically — it uses `.view()` when possible, and silently copies the data (equivalent to `.contiguous()`) when not.

```python
x.transpose(1, 2).view(...)      # may error if not contiguous
x.transpose(1, 2).reshape(...)   # always works
```

Rule of thumb: use `.reshape()` by default. There's no real downside — if the tensor is already contiguous, it behaves exactly like `.view()`, no copy involved.

---

## Summary

| Operation | Direction | Example |
|---|---|---|
| Flatten logits for loss | merge `(B, T, V)` -> `(B*T, V)` | cross-entropy over all tokens |
| Split into attention heads | split `(B, T, C)` -> `(B, n_heads, T, head_dim)` | parallel attention |
| Merge heads back | merge `(B, n_heads, T, head_dim)` -> `(B, T, C)` | output projection |

Same underlying operation (reshape), just applied in opposite directions depending on what the next step in the computation needs.

In [11]:
import torch


Q = torch.arange(0, 10)

print(f'Q before: {Q}')

Q before: tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])


In [12]:
Q = Q.reshape(1, 2, 5)
print(f"Q after:\n{Q}")
print(Q.shape)

Q after:
tensor([[[0, 1, 2, 3, 4],
         [5, 6, 7, 8, 9]]])
torch.Size([1, 2, 5])


In [ ]:
Q = Q.transpose(1, 2)
print(f"Q after:\n{Q}")
print(Q.shape)

Q after:
tensor([[[0, 5],
         [1, 6],
         [2, 7],
         [3, 8],
         [4, 9]]])
torch.Size([1, 5, 2])


In [30]:
import torch.nn.functional as F

logits = F.softmax(torch.rand((4, 2, 4)), dim =-1)
print(f'=' * 50)
print(f'Logits before:\n{logits}')
print('Here each row its a token and each column its the value for the prob of next token')

B, T, C = logits.shape
logits = logits.reshape(B*T, C)

print(f'=' * 50)
print(f'Logits after:\n{logits}')
print('Here each row its a token and each column its the value for the prob of next toke too, but now the batch flatten in only one')

Logits before:
tensor([[[0.3273, 0.3189, 0.1426, 0.2112],
         [0.3411, 0.2180, 0.3072, 0.1338]],

        [[0.2083, 0.1840, 0.2905, 0.3172],
         [0.2657, 0.1850, 0.3355, 0.2138]],

        [[0.2331, 0.3412, 0.2376, 0.1881],
         [0.2863, 0.2299, 0.1939, 0.2900]],

        [[0.1977, 0.1735, 0.3757, 0.2531],
         [0.1667, 0.2966, 0.1769, 0.3597]]])
Here each row its a token and each column its the value for the prob of next token
Logits after:
tensor([[0.3273, 0.3189, 0.1426, 0.2112],
        [0.3411, 0.2180, 0.3072, 0.1338],
        [0.2083, 0.1840, 0.2905, 0.3172],
        [0.2657, 0.1850, 0.3355, 0.2138],
        [0.2331, 0.3412, 0.2376, 0.1881],
        [0.2863, 0.2299, 0.1939, 0.2900],
        [0.1977, 0.1735, 0.3757, 0.2531],
        [0.1667, 0.2966, 0.1769, 0.3597]])
Here each row its a token and each column its the value for the prob of next toke too, but now the batch flatten in only one


In [45]:
Q = torch.arange(1, 257)
print( Q.shape)

Q = Q.reshape(1, 8, 32)
print(f'=' * 50)
print(Q, Q.shape)


Q = Q.transpose(1,2 )
print(f'=' * 50)
print(Q, Q.shape)

Q = Q.transpose(1, 2).reshape(8 * 32)
print(f'=' * 50)
print(Q, Q.shape)


torch.Size([256])
tensor([[[  1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,  14,
           15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,  27,  28,
           29,  30,  31,  32],
         [ 33,  34,  35,  36,  37,  38,  39,  40,  41,  42,  43,  44,  45,  46,
           47,  48,  49,  50,  51,  52,  53,  54,  55,  56,  57,  58,  59,  60,
           61,  62,  63,  64],
         [ 65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,  78,
           79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,  91,  92,
           93,  94,  95,  96],
         [ 97,  98,  99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110,
          111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124,
          125, 126, 127, 128],
         [129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142,
          143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156,
          157, 158, 159, 160],
         [161, 162, 163, 16